# spaCy: fundamentos para analizar texto en español

**Duración estimada:** 35 minutos

## Propósito
En este notebook vas a practicar las operaciones básicas de `spaCy` sobre ejemplos breves. La idea es que puedas mirar una salida, interpretarla y decidir para qué sirve.

## Cómo trabajar este material
- Ejecutá cada bloque y frena a leer la salida antes de avanzar.
- Si usas un asistente de IA, pedile hipótesis o explicaciones alternativas, pero contrastalas siempre con lo que devuelve el código.

## Resultados de aprendizaje
Al finalizar, vas a poder:
- reconocer tokens, lemas y etiquetas gramaticales;
- leer una visualización de dependencias y de entidades;
- identificar grupos nominales;
- comprender, a nivel introductorio, como se agrega una personalización simple al pipeline.


In [ ]:
!pip install spacy -q
!python -m spacy download es_core_news_sm -q


In [ ]:
import spacy
from collections import Counter
from spacy import displacy

nlp = spacy.load("es_core_news_sm")


## Caso 1: tokenización, lema y categoría gramatical

En este primer caso vamos a observar cómo `spaCy` divide una oración y que información le asigna a cada token.

**Qué deberías mirar:**
- dónde corta los tokens;
- qué lema propone para cada palabra;
- qué etiquetas POS aparecen con más frecuencia.


In [ ]:
doc = nlp("Yo, Matías Barreto, estoy volando hacia Buenos Aires.")
print([token.text for token in doc])


In [ ]:
print("La cantidad de tokens es:", len(doc))


In [ ]:
print("Tokens del documento")
print("_" * 20)
for token in doc:
    print(token)


In [ ]:
for token in doc:
    print(token.text, token.lemma_)


In [ ]:
for token in doc:
    print(token.text, token.pos_, spacy.explain(token.pos_))


In [ ]:
frecuencias = Counter(token.text for token in doc)
print(frecuencias)


**Pausa de lectura**

En este caso conviene diferenciar tres niveles:
- el texto original del token (`token.text`);
- la forma base (`token.lemma_`);
- la función gramatical (`token.pos_`).

Esa distinción va a ser clave cuando trabajes con noticias o corpus más grandes.


## Caso 2: lemas y dependencias sintácticas

Ahora vamos a mirar relaciones entre palabras. Las dependencias sirven para ver cómo se organiza una oración y que vínculos gramaticales aparecen.


In [ ]:
doc = nlp("En abril de 2026 estaremos realizando una introducción a spaCy para trabajar conceptos básicos de PLN.")
for token in doc:
    print(token.text, token.lemma_)


In [ ]:
from spacy.lang.es.examples import sentences

doc = nlp(sentences[11])
print(doc.text)
for token in doc:
    print(token.text, token.pos_, token.dep_)


In [ ]:
conteo_pos = Counter(token.pos_ for token in doc)
print(conteo_pos)


In [ ]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})


**Preguntas para guiar tu lectura**

- Cuál parece ser el verbo principal de la oración?
- Qué palabras dependen de ese verbo?
- Qué información te aporta la visualización que no veías tan rápido en la lista de tokens?


## Caso 3: entidades nombradas

En este caso vamos a mirar cómo `spaCy` detecta nombres propios, lugares y otras entidades relevantes.


In [ ]:
text = "Queremos viajar desde Buenos Aires a Mar del Plata y, unos días después, a La Plata."
doc = nlp(text)
displacy.render(doc, style='ent', jupyter=True, options={'distance': 200})


In [ ]:
text = "Su nombre es Benjamín; nació en Argentina."
doc = nlp(text)
displacy.render(doc, style='ent', jupyter=True, options={'distance': 200})


In [ ]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})


**Importante**

Los modelos no aciertan siempre. Si una entidad te parece discutible, no la des por válida automáticamente: usala como punto de análisis.


## Caso 4: grupos nominales

Los grupos nominales (`noun_chunks`) te ayudan a detectar segmentos como `"mis padres"` o `"Buenos Aires"` cuando funcionan como unidades dentro de la oración.


In [ ]:
for token in nlp("Mis padres y amigos viven en Buenos Aires."):
    print(token.text)


In [ ]:
for chunk in nlp("Mis padres y amigos viven en Buenos Aires.").noun_chunks:
    print(chunk.text)


En este punto conviene comparar la lista completa de tokens con la lista de grupos nominales: no muestran lo mismo ni sirven para la misma tarea.


## Caso 5: extensión opcional - personalizar la lematización

Este caso es más avanzado. No hace falta dominarlo ahora, pero sirve para entender que el pipeline también se puede adaptar a necesidades puntuales.


In [ ]:
doc = nlp("Estoy volando hacia Baires")
print([token.text for token in doc])


In [ ]:
from spacy.language import Language


In [ ]:
@Language.component("custom_lemmatizer")
def custom_lemmatizer(doc):
    for token in doc:
        if token.text == "Baires":
            token.lemma_ = "Buenos Aires"
    return doc

if "custom_lemmatizer" in nlp.pipe_names:
    nlp.remove_pipe("custom_lemmatizer")

nlp.add_pipe("custom_lemmatizer", after="lemmatizer")

doc = nlp("Estoy volando a Baires")
print([token.text for token in doc])
print([token.lemma_ for token in doc])


## Cierre y continuidad

Antes de pasar al siguiente notebook, comprobá si podés responder estas preguntas sin mirar las celdas anteriores:
- Qué diferencia hay entre token, lema y POS?
- Para qué sirve mirar dependencias?
- Por qué una entidad detectada por el modelo no siempre debe aceptarse sin revisión?

**Actividad breve con IA:** pedile a un asistente que te proponga una oración y que anticipe cómo la tokenizaría `spaCy`. Después corré esa oración en el notebook y compará ambas lecturas.
